# Hartford County Grid Restoration — Mathematical Methodology

This notebook traces **exactly** how each factor in the simulation alters the restoration outcome, step by step. Every formula below has a direct counterpart in the JavaScript front-end (`03_grid_simulation.html`) and the Python scheduler (`07_server.py`).

**Sections**
1. Storm Outage Placement (wind-weighted sampling)
2. Repair Time Distribution (log-normal + environmental multipliers)
3. Crew Dispatch (priority scoring, travel time, specialization)
4. End-to-End Mini Simulation (5 outages · 2 crews · full step trace)
5. Restoration Curve & SAIDI
6. Monte Carlo Ensemble (30-seed uncertainty bands)

In [ ]:
import math, numpy as np, matplotlib.pyplot as plt
from collections import namedtuple

np.random.seed(42)
plt.rcParams.update({'figure.figsize': (12, 4), 'axes.grid': True, 'grid.alpha': 0.3})
print('Ready.')

---
## 1  Storm Outage Placement

The network is divided into **line segments** (feeder sections + laterals).  
Each segment $s$ has a midpoint $(lat_s, lon_s)$ and a population served $C_s$.

### 1a  Wind-speed weighting (HRRR path)

The 10-m wind speed $v(s)$ is bilinearly interpolated from the 15 × 21 Hartford County HRRR grid.  Outage weight:

$$w(s) = \max\!\bigl(0.05,\;(v(s)/30)^{2}\bigr)$$

The quadratic reflects kinetic energy ($\propto v^2$).  Floor 0.05 keeps every segment in play (aging hardware, local vulnerabilities).

### 1b  Gaussian track-decay fallback

When HRRR data is unavailable (pre-2014 storms):

$$w(s) = \max\!\bigl(0.1,\;(v_\text{track}/50)\cdot e^{-d(s)^2/(2\cdot 30^2)}\bigr)$$

$d(s)$ = haversine distance from segment midpoint to nearest HURDAT2 track point (mi).  
$\sigma = 30$ mi captures the typical tropical-cyclone wind-field radius for CT storms.

### 1c  Weighted random sampling

$N$ outage locations are drawn **without replacement** from the cumulative weight distribution:

$$P(\text{outage on }s) = w(s)\;/\;\textstyle\sum_j w(j)$$

In [ ]:
# ── Wind-weighting worked example ─────────────────────────────────
seg_names = ['Hartford downtown', 'West Hartford suburban', 'Granby rural', 'Farmington corridor']
wind_mph  = np.array([14.0, 17.5, 21.8, 19.2])   # from HRRR grid at each segment midpoint

# Equation: w(s) = max(0.05, (v/30)^2)
weights = np.maximum(0.05, (wind_mph / 30) ** 2)
probs   = weights / weights.sum()

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].bar(range(4), wind_mph, color="steelblue")
axes[0].set_xticks(range(4)); axes[0].set_xticklabels(seg_names, rotation=15, ha="right", fontsize=8)
axes[0].set_ylabel("10-m wind speed (mph)"); axes[0].set_title("Input: HRRR wind field")

axes[1].bar(range(4), weights, color="darkorange")
axes[1].set_xticks(range(4)); axes[1].set_xticklabels(seg_names, rotation=15, ha="right", fontsize=8)
axes[1].set_ylabel("w(s) = max(0.05, (v/30)²)"); axes[1].set_title("Outage weight per segment")

axes[2].bar(range(4), probs * 100, color="seagreen")
axes[2].set_xticks(range(4)); axes[2].set_xticklabels(seg_names, rotation=15, ha="right", fontsize=8)
axes[2].set_ylabel("P(outage at segment) %"); axes[2].set_title("Normalized sampling probability")
for i, p in enumerate(probs):
    axes[2].text(i, p*100 + 0.3, f"{p*100:.1f}%", ha="center", fontsize=9)

plt.tight_layout(); plt.show()

print("Segment-by-segment breakdown:")
print(f"  {'Segment':<28} {'wind(mph)':>10} {'weight':>8} {'P(outage)':>10}")
for n, v, w, p in zip(seg_names, wind_mph, weights, probs):
    print(f"  {n:<28} {v:>10.1f} {w:>8.4f} {p*100:>9.1f}%")

---
## 2  Repair Time Distribution

Each outage $i$ gets an independent repair time from a **log-normal** distribution:

$$\text{repairH}_i = \text{clamp}_{[0.25,\,12]}\!\bigl(e^{\,\mu + \sigma\varepsilon_i}\bigr), \quad \varepsilon_i \sim \mathcal{N}(0,1)$$

| Parameter | Value | Meaning |
|---|---|---|
| $\mu$ | $\ln 2 = 0.693$ | Median repair = 2 hours |
| $\sigma$ | $0.857$ | Calibrated to empirical CT outage data (Wanik et al. 2017) |
| Range | [0.25 h, 12 h] | 15-min minimum; 12-h cap for complex damage |

### 2b  Environmental multipliers (from HRRR)

$$\text{repairH}_i \;\leftarrow\; \text{repairH}_i \times M_T \times M_P \times M_E(p) \times M_F(d)$$

**Temperature** $M_T$: cold slows field crews (tool handling, footing, PPE).  
**Precipitation** $M_P$: ice adds re-sag risk, hazard-zone clearance time.  
**Equipment shortage** $M_E$: spare transformers/wire become scarce after >60 % restored.  
**Fatigue** $M_F$: performance degrades after day 2 of continuous shifts.

| Factor | Condition | $M$ |
|---|---|---|
| $M_T$ | $T \geq 50°F$ | 1.00 |
| $M_T$ | $32°F \leq T < 50°F$ | 1.15 |
| $M_T$ | $T < 32°F$ | 1.35 |
| $M_P$ | Rain | 1.00 |
| $M_P$ | Mix | 1.10 |
| $M_P$ | Snow | 1.20 |
| $M_P$ | Ice | 1.45 |
| $M_E$ | $p > 0.6$ | $1 + 0.4\cdot(p-0.6)/0.4$ |
| $M_F$ | days on shift $d > 2$ | $1 + \min(0.30,\;0.05(d-2))$ |

In [ ]:
# ── Repair time distribution ──────────────────────────────────────
MU, SIGMA = np.log(2), 0.857
raw    = np.random.lognormal(mean=MU, sigma=SIGMA, size=20_000)
clamped = np.clip(raw, 0.25, 12)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: distribution shape
axes[0].hist(clamped, bins=80, density=True, color="steelblue", alpha=0.7)
for p, col, ls in [(50,"red","--"), (75,"orange","-."), (90,"crimson",":")]:
    v = np.percentile(clamped, p)
    axes[0].axvline(v, color=col, ls=ls, lw=1.8, label=f"P{p} = {v:.2f} h")
axes[0].set_xlabel("Repair hours"); axes[0].set_ylabel("Density")
axes[0].set_title(r"Base repair time: $e^{\mu+\sigma\varepsilon}$ clamped to [0.25, 12] h")
axes[0].legend()

# Right: four weather scenarios
scenarios = {
    "Summer rain\n(Isaias-like)":   (1.00, 1.00, "steelblue"),
    "Cold mix\n(Jan 2024)":         (1.15, 1.10, "goldenrod"),
    "Freezing snow\n(Dec 2023)":    (1.35, 1.20, "coral"),
    "Ice storm\n(Snowtober-like)":  (1.35, 1.45, "crimson"),
}
base = np.median(clamped)
labels, medians, colors = [], [], []
for lbl, (mt, mp, col) in scenarios.items():
    labels.append(lbl); colors.append(col)
    medians.append(base * mt * mp)

axes[1].bar(range(len(scenarios)), medians, color=colors, alpha=0.85)
axes[1].axhline(base, color="steelblue", ls="--", lw=1.5, label=f"Base median {base:.2f} h")
axes[1].set_xticks(range(len(scenarios)))
axes[1].set_xticklabels(labels, fontsize=8)
axes[1].set_ylabel("Effective median repair time (h)")
axes[1].set_title("Repair time × temperature × precipitation multipliers")
axes[1].legend(fontsize=8)
for i, (m, (lbl, (mt, mp, _))) in enumerate(zip(medians, scenarios.items())):
    axes[1].text(i, m + 0.04, f"{m:.2f} h\n(×{mt*mp:.2f})", ha="center", fontsize=8)

plt.tight_layout(); plt.show()

print("Base repair time percentiles:")
for p in [25, 50, 75, 90, 95]:
    print(f"  P{p:2d}: {np.percentile(clamped, p):.2f} h")

---
## 3  Crew Dispatch — Priority, Travel Time, Specialization

### 3a  Priority score

At each dispatch event, every unassigned discovered outage $i$ receives:

$$\text{score}(i) = \alpha\,C_i + 10^{6}\,\mathbf{1}[\text{make-safe}(i)] + 10^{4}\,\mathbf{1}[\text{critical}(i)]$$

where $\alpha$ = customer-weight parameter (0–2, user-configurable) and $C_i$ = customers affected.  The large bonuses guarantee make-safe hazards (live wires) and critical facilities (hospitals, fire stations) are always dispatched first.

### 3b  Travel time

$$t_\text{travel}(i,c) = d_\text{haversine}(i,c)\times R_\text{road}\;/\;v_\text{field}$$

| Mode | $v_\text{field}$ | $R_\text{road}$ |
|---|---|---|
| Realistic | 25 mph | 1.50 |
| Realistic + soil-saturated | 25 mph | 1.875 |
| Non-realistic | 45 mph | 1.00 |

### 3c  Crew specialization

30% of outages require **tree-clearing** before the line can be repaired.  Only tree-qualified crews handle these; line-only crews skip them.  The 80/20 line-to-tree crew ratio is configurable.

### 3d  Assignment rule

The **greedy nearest-available** rule assigns the highest-priority discovered outage to the earliest-free qualified crew:

$$i^* = \arg\max_i\,\text{score}(i), \quad c^* = \arg\min_{c \in \mathcal{A}(t)}\,t_\text{travel}(i^*,c)$$

Crew $c^*$ is then busy until $t_\text{free}(c^*) = t_\text{arrive}(i^*) + \text{repairH}_{i^*}$.

In [ ]:
# ── Priority + travel-time example ───────────────────────────────
def haversine_mi(lat1, lon1, lat2, lon2):
    R = 3958.8
    dlat, dlon = math.radians(lat2-lat1), math.radians(lon2-lon1)
    a = math.sin(dlat/2)**2 + math.cos(math.radians(lat1))*math.cos(math.radians(lat2))*math.sin(dlon/2)**2
    return R * 2 * math.asin(math.sqrt(a))

ALPHA   = 1.0   # customer weight (user-configurable)
V_FIELD = 25    # mph (realistic mode)
R_ROAD  = 1.5   # road multiplier

Outage = namedtuple("Outage", ["id","lat","lon","customers","critical","make_safe","tree_req"])
outages = [
    Outage(1, 41.76, -72.68, 1800, False, False, False),  # Hartford downtown
    Outage(2, 41.90, -72.75,  320, True,  False, True ),  # Critical + tree-clearing
    Outage(3, 41.55, -72.88, 2200, False, False, False),  # Large suburban
    Outage(4, 41.82, -72.54,  150, False, True,  False),  # Make-safe hazard
    Outage(5, 42.00, -72.62,  900, True,  False, False),  # Critical facility
]

def score(o):
    return ALPHA*o.customers + 1_000_000*o.make_safe + 10_000*o.critical

DEPOT_LAT, DEPOT_LON = 41.76, -72.68  # crew starting position

print(f"Priority scores (alpha={ALPHA}):")
print(f"  {'#':>3}  {'Customers':>10}  {'Critical':>8}  {'MakeSafe':>8}  {'Score':>12}  {'Travel(min)':>12}")
for o in sorted(outages, key=score, reverse=True):
    d_mi = haversine_mi(DEPOT_LAT, DEPOT_LON, o.lat, o.lon)
    t_min = d_mi * R_ROAD / V_FIELD * 60
    print(f"  {o.id:>3}  {o.customers:>10,}  {str(o.critical):>8}  {str(o.make_safe):>8}",
          f"  {score(o):>12,.0f}  {t_min:>11.0f}")

---
## 4  End-to-End Mini Simulation

We trace every arithmetic step for **5 outages and 2 crews**.  
Repair times are fixed (deterministic) for clarity; in the real simulation they are drawn from the log-normal above.

**Timeline** (hours from storm end):

$$t_\text{discover}(i) = t_\text{assess} + \delta_i$$
$$t_\text{dispatch}(i,c) = \max\bigl(t_\text{free}(c),\; t_\text{discover}(i)\bigr)$$
$$t_\text{arrive}(i,c) = t_\text{dispatch}(i,c) + t_\text{travel}(i,c)$$
$$t_\text{finish}(i,c) = t_\text{arrive}(i,c) + \text{repairH}_i$$

`t_assess = 12 h` (post-storm assessment delay; eliminated by pre-staging flag).  
Discovery offset $\delta_i$: 30 % of outages visible in hour 1 (AMI/smart-meters);  
remaining 70 % revealed over next 36 h via customer calls.

In [ ]:
# ── Deterministic mini-sim trace ─────────────────────────────────
T_ASSESS = 12.0   # hours of post-storm damage assessment

MiniOut = namedtuple("MiniOut", ["id","lat","lon","customers","critical","tree_req","repair_h","delta_h"])
mini_outs = [
    MiniOut(1, 41.76,-72.68, 1800, False, False, 2.5, 0.5),  # seen early (AMI)
    MiniOut(2, 41.90,-72.75,  320, True,  True,  5.0, 1.0),  # critical + tree
    MiniOut(3, 41.55,-72.88, 2200, False, False, 1.8, 3.0),  # large, found later
    MiniOut(4, 41.82,-72.54,  150, False, False, 3.2, 8.0),  # found much later
    MiniOut(5, 42.00,-72.62,  900, True,  False, 4.1, 2.0),  # critical
]

Crew = namedtuple("Crew", ["id","lat","lon"])
crews = [Crew("Line-A", 41.76,-72.70), Crew("Tree-B", 41.80,-72.65)]

crew_free = {c.id: T_ASSESS for c in crews}
crew_pos  = {c.id: (c.lat, c.lon) for c in crews}
assigned  = set()
log       = []

def mini_score(o):
    return ALPHA*o.customers + 10_000*o.critical

print("Step  Outage  Crew    t_discover  t_dispatch  t_travel   t_arrive  repair   t_finish  Customers")
print("-"*105)

for step in range(len(mini_outs)):
    t_now = min(crew_free.values())
    # Available: discovered and not yet assigned
    avail = [o for o in mini_outs if o.id not in assigned and T_ASSESS+o.delta_h <= t_now]
    if not avail:
        t_now = min(T_ASSESS + o.delta_h for o in mini_outs if o.id not in assigned)
        avail = [o for o in mini_outs if o.id not in assigned and T_ASSESS+o.delta_h <= t_now]
    if not avail: break

    best  = max(avail, key=mini_score)
    crew  = min([c for c in crews if crew_free[c.id] <= t_now], key=lambda c: crew_free[c.id])
    clat, clon = crew_pos[crew.id]
    t_disc     = T_ASSESS + best.delta_h
    t_disp     = max(crew_free[crew.id], t_disc)
    t_trav     = haversine_mi(clat, clon, best.lat, best.lon) * R_ROAD / V_FIELD
    t_arr      = t_disp + t_trav
    t_fin      = t_arr + best.repair_h

    assigned.add(best.id)
    crew_free[crew.id] = t_fin
    crew_pos[crew.id]  = (best.lat, best.lon)
    log.append(dict(outage=best.id, crew=crew.id, customers=best.customers,
                    critical=best.critical, t_disc=t_disc, t_disp=t_disp,
                    t_trav=t_trav, t_arr=t_arr, repair=best.repair_h, t_fin=t_fin))

    print(f"  {step+1:>3}   {best.id:>5}  {crew.id:<8}",
          f"  {t_disc:>9.2f}h  {t_disp:>9.2f}h  {t_trav*60:>7.0f}m",
          f"  {t_arr:>8.2f}h  {best.repair_h:>6.1f}h  {t_fin:>8.2f}h  {best.customers:>9,}")

total_c = sum(o.customers for o in mini_outs)
saidi   = sum(r["customers"]*r["t_fin"] for r in log) / total_c
print("-"*105)
print(f"Total customers: {total_c:,}")
print(f"Last outage restored at: {max(r['t_fin'] for r in log):.2f} h")
print(f"SAIDI = sum(Ci * t_finish_i) / C_total = {saidi:.3f} customer-h/customer")
print(f"      = {saidi*60:.1f} customer-minutes/customer")

---
## 5  Restoration Curve & SAIDI

The **restoration curve** $f(t)$ is the fraction of customers still without power:

$$f(t) = \frac{\sum_i C_i \cdot \mathbf{1}[t_\text{finish}(i) > t]}{\sum_i C_i}$$

**SAIDI** (System Average Interruption Duration Index):

$$\text{SAIDI} = \sum_i \frac{C_i \cdot t_\text{finish}(i)}{C_\text{total}}$$

SAIDI = area under the restoration curve (customer-hours per customer).  
A SAIDI of 20 means the average customer was without power for 20 hours.  
DOE OE-417 events let us calibrate simulated SAIDI against real CT outage history.

In [ ]:
# ── Restoration curve from mini-sim log ──────────────────────────
t_max  = max(r["t_fin"] for r in log) * 1.05
t_grid = np.linspace(0, t_max, 500)
f_t    = np.array([sum(r["customers"] for r in log if r["t_fin"] > t) / total_c
                   for t in t_grid])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: restoration curve
axes[0].step(t_grid, f_t*100, where="post", color="steelblue", lw=2.5)
axes[0].fill_between(t_grid, f_t*100, step="post", alpha=0.12, color="steelblue")
axes[0].set_xlabel("Hours since storm end")
axes[0].set_ylabel("% customers without power")
axes[0].set_title("Restoration Curve (mini example, 5 outages)")
axes[0].set_ylim(0, 110)
for r in sorted(log, key=lambda x: x["t_fin"]):
    pct = sum(rr["customers"] for rr in log if rr["t_fin"] > r["t_fin"]) / total_c * 100
    axes[0].annotate(f"Outage {r['outage']}\n{r['customers']:,} cust",
                     xy=(r["t_fin"], pct), xytext=(r["t_fin"]+0.4, pct+6),
                     fontsize=7.5, arrowprops=dict(arrowstyle="->", color="gray", lw=0.8))

# Right: SAIDI breakdown
saidi_contrib = {f"Outage {r['outage']}": r["customers"]*r["t_fin"]/total_c for r in log}
axes[1].barh(list(saidi_contrib.keys()), list(saidi_contrib.values()), color="darkorange", alpha=0.85)
axes[1].axvline(saidi, color="red", ls="--", lw=1.8, label=f"Total SAIDI = {saidi:.2f} h")
axes[1].set_xlabel("SAIDI contribution (customer-h / customer)")
axes[1].set_title("SAIDI contribution by outage")
axes[1].legend()

plt.tight_layout(); plt.show()

# Numeric check: SAIDI = area under restoration curve
saidi_numeric = np.trapz(f_t, t_grid)
print(f"SAIDI (formula):       {saidi:.4f} h")
print(f"SAIDI (curve area):    {saidi_numeric:.4f} h  ← should match")

---
## 6  Monte Carlo Ensemble (30 Seeds)

A single simulation run is one **stochastic draw** — a plausible scenario,  
not a forecast.  The 30-seed ensemble quantifies uncertainty:

$$\bar{f}(t) = \frac{1}{30}\sum_{k=1}^{30} f_k(t)$$
$$\text{Band}_{10\text{–}90}(t) = \bigl[f_{10\%}(t),\; f_{90\%}(t)\bigr]$$

The band width measures **aleatory uncertainty** — genuine randomness in which  
segments fail and how long each repair takes.

| Source of randomness | Distribution |
|---|---|
| Outage location | Weighted-random over $w(s)$ |
| Repair time | Log-normal $(\mu=\ln 2,\;\sigma=0.857)$ |
| Tree-clearing requirement | Bernoulli(0.30) per outage |
| Customer callback lag | Exponential (when AMI coverage off) |
| Crew fatigue accumulation | Deterministic from dispatch timeline |

In [ ]:
# ── 30-seed restoration-curve ensemble ───────────────────────────
def run_seed(seed, n_out=20, n_crew=4, assess=12.0):
    rng = np.random.default_rng(seed)
    repair   = np.clip(rng.lognormal(np.log(2), 0.857, n_out), 0.25, 12)
    cust     = rng.lognormal(np.log(1200), 0.9, n_out).astype(int)
    disc_off = np.where(rng.random(n_out) < 0.30,
                        rng.uniform(0, 1, n_out),
                        1 + rng.exponential(10, n_out))
    # Equipment shortage: repair × (1 + 0.4*max(0, p-0.6)/0.4) where p = fraction done
    crew_free = np.full(n_crew, assess)
    finish    = np.zeros(n_out)
    order     = np.argsort(-cust)      # highest-customer first (priority dispatch)
    n_done    = 0
    for idx in order:
        t_disp  = max(crew_free.min(), assess + disc_off[idx])
        ci      = crew_free.argmin()
        p_done  = n_done / n_out
        me      = 1 + 0.4 * max(0, p_done - 0.6) / 0.4 if p_done > 0.6 else 1.0
        finish[idx]   = t_disp + repair[idx] * me
        crew_free[ci] = finish[idx]
        n_done += 1
    return finish, cust

N_SEEDS = 30
t_eval  = np.linspace(0, 80, 400)
curves  = []
for s in range(N_SEEDS):
    ft, cu = run_seed(s)
    total  = cu.sum()
    curves.append(np.array([cu[ft > t].sum() / total for t in t_eval]) * 100)
curves = np.array(curves)

mean_c = curves.mean(0)
p10, p50, p90 = np.percentile(curves, [10, 50, 90], axis=0)

fig, ax = plt.subplots(figsize=(12, 5))
for i in range(N_SEEDS):
    ax.plot(t_eval, curves[i], color="gray", lw=0.5, alpha=0.3)
ax.fill_between(t_eval, p10, p90, alpha=0.2, color="steelblue", label="P10–P90 band")
ax.plot(t_eval, mean_c, "steelblue", lw=2.5, label="Ensemble mean")
ax.plot(t_eval, p50,    "steelblue", lw=1.5, ls="--", label="Ensemble median")
ax.set_xlabel("Hours since storm end"); ax.set_ylabel("% customers without power")
ax.set_title(f"Monte Carlo Ensemble ({N_SEEDS} seeds)  —  Restoration Curve")
ax.legend(fontsize=10); ax.set_ylim(0, 108)
plt.tight_layout(); plt.show()

# Time-to-50%-restoration across seeds
t50 = []
for c in curves:
    idx = np.where(c <= 50)[0]
    t50.append(t_eval[idx[0]] if len(idx) else t_eval[-1])
t50 = np.array(t50)
print("Time to 50% restoration (across 30 seeds):")
for pct, val in [(10, np.percentile(t50,10)), (50, np.median(t50)),
                 (90, np.percentile(t50,90)), ("mean", t50.mean())]:
    print(f"  {str(pct):>5}: {val:.1f} h")